In [ ]:
import pandas as pd

# Load the CSV file
csv_file_path = './modelC.csv'
df_csv = pd.read_csv(csv_file_path)

df = df_csv
print("CSV DataFrame loaded successfully:")
# print(df_csv.head())


CSV DataFrame loaded successfully:


In [ ]:
df.head()

In [ ]:
(df['system_a'].unique())
print(df['system_b'].unique())

In [ ]:
df.columns

In [5]:
# Equal-weight sum of all difference scores
score_prefixes = [
    'pred_streams', 'pred_likes', 'pred_coherence', 'pred_musicality',
    'pred_memorability', 'pred_clarity', 'pred_naturalness',
]
for prefix in score_prefixes:
    df[f'diff_{prefix}'] = df[f'{prefix}_a'] - df[f'{prefix}_b']

diff_cols = [f'diff_{p}' for p in score_prefixes]

# Normalize each diff to z-score first (so scales don't dominate)
from scipy.stats import zscore
df_z = df[diff_cols].apply(zscore)

df['combined_equal'] = df_z.sum(axis=1)
acc_equal = ((df['combined_equal'] > 0) == (df['user_preference'] == 'A')).mean()
print(f"Equal-weight combined: {acc_equal:.3f}")

# Also try likes+streams only (your two significant ones)
df['combined_sig'] = df_z[['diff_pred_likes', 'diff_pred_streams']].sum(axis=1)
acc_sig = ((df['combined_sig'] > 0) == (df['user_preference'] == 'A')).mean()
print(f"Likes+Streams only:    {acc_sig:.3f}")

# And raw likes alone for reference
df['diff_likes_raw'] = df['pred_likes_a'] - df['pred_likes_b']
acc_likes = ((df['diff_likes_raw'] > 0) == (df['user_preference'] == 'A')).mean()
print(f"Likes alone (baseline):{acc_likes:.3f}")

Equal-weight combined: 0.454
Likes+Streams only:    0.488
Likes alone (baseline):0.503


In [6]:
def conditional_pred(row):
    if row['is_instrumental']:
        return row['model_pref_likes']  # or combined_popularity
    else:
        return row['model_pref_coherence']  # best non-instrumental candidate

df['model_pref_conditional'] = df.apply(conditional_pred, axis=1)
df['correct_conditional'] = (df['model_pref_conditional'] == df['user_preference'])

overall = df['correct_conditional'].mean()
instr = df[df['is_instrumental']]['correct_conditional'].mean()
non_instr = df[~df['is_instrumental']]['correct_conditional'].mean()

print(f"Conditional rule:  overall={overall:.3f}  instr={instr:.3f}  non_instr={non_instr:.3f}")


Conditional rule:  overall=0.506  instr=0.529  non_instr=0.468


In [7]:
# Is non-instrumental accuracy uniform across prompts, or are some prompt types okay?
non_instr = df[~df['is_instrumental']].copy()

print("=== By prompt (top 15) ===")
prompt_acc = (non_instr.groupby('prompt')['correct_likes']
              .agg(['mean', 'count'])
              .sort_values('mean', ascending=False))
print(prompt_acc[prompt_acc['count'] >= 5].head(15).round(3))

print("\n=== By system pair ===")
non_instr['system_pair'] = non_instr['system_a'] + ' vs ' + non_instr['system_b']
pair_acc = (non_instr.groupby('system_pair')['correct_likes']
            .agg(['mean', 'count'])
            .sort_values('mean', ascending=False))
print(pair_acc[pair_acc['count'] >= 5].round(3))

=== By prompt (top 15) ===
                                                    mean  count
prompt                                                         
1970s roots reggae with a deep dub groove, slow...   0.6     10

=== By system pair ===
                                                mean  count
system_pair                                                
acestep-1.5-turbo-1.7b vs sonauto-v3-preview   0.818     11
acestep-1.5-turbo-1.7b vs lyria-3-30s          0.773     22
sonauto-v2-2 vs acestep-1.5-turbo-1.7b         0.706     17
lyria-3-30s vs elevenlabs-music-v1             0.684     19
sonauto-v2-2 vs elevenlabs-music-v1            0.577     26
lyria-3-30s vs sonauto-v2-2                    0.556     27
elevenlabs-music-v1 vs acestep-1.5-turbo-1.7b  0.529     17
lyria-3-30s vs acestep-1.5-turbo-1.7b          0.526     19
sonauto-v2-2 vs riffusion-fuzz-1-1             0.500     10
elevenlabs-music-v1 vs lyria-3-30s             0.464     28
elevenlabs-music-v1 vs riffusion-fuzz

In [214]:
# Is sonauto-v3 overrepresented in non-instrumental?
df['involves_v3'] = df['system_a'].str.contains('sonauto-v3') | df['system_b'].str.contains('sonauto-v3')

print(df.groupby('is_instrumental')['involves_v3'].mean().round(3))
print(df.groupby('is_instrumental')['involves_v3'].sum())

# What does accuracy look like if we just exclude v3 battles?
mask_no_v3 = ~df['involves_v3']
print(f"\nExcluding sonauto-v3 battles:")
print(f"  N remaining: {mask_no_v3.sum()}")
print(f"  overall:         {df[mask_no_v3]['correct_likes'].mean():.3f}")
print(f"  instrumental:    {df[mask_no_v3 & df['is_instrumental']]['correct_likes'].mean():.3f}")
print(f"  non-instrumental:{df[mask_no_v3 & ~df['is_instrumental']]['correct_likes'].mean():.3f}")

is_instrumental
False    0.171
True     0.083
Name: involves_v3, dtype: float64
is_instrumental
False    82
True     65
Name: involves_v3, dtype: int64

Excluding sonauto-v3 battles:
  N remaining: 1112
  overall:         0.482
  instrumental:    0.473
  non-instrumental:0.499


# 10 CV code with feature interactions:

In [11]:
from pyexpat import features
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.metrics import f1_score, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")


# Stratified
instr_mask = df['is_instrumental'].values
# print(f"Instrumental:        {accuracy_score(y[instr_mask], oof_preds[instr_mask]):.3f}")
# print(f"Non-instrumental:    {accuracy_score(y[~instr_mask], oof_preds[~instr_mask]):.3f}")



score_prefixes = [
    'pred_streams', 'pred_likes', 'pred_coherence', 'pred_musicality',
    'pred_memorability', 'pred_clarity', 'pred_naturalness',
    'combined_popularity', 'combined_songeval', 'combined_overall'
]
score_prefixes = [
    'pred_streams', 'pred_likes',
    'combined_popularity'
]
for prefix in score_prefixes:
    df[f'diff_{prefix}'] = df[f'{prefix}_a'] - df[f'{prefix}_b']

diff_cols = [f'diff_{p}' for p in score_prefixes]

# Create instrumental feature and interactions
df['is_instrumental_int'] = df['is_instrumental'].astype(int)

for prefix in score_prefixes:
    df[f'diff_{prefix}_x_instr'] = df[f'diff_{prefix}'] * df['is_instrumental_int']

# Ratio instead of difference — captures relative rather than absolute gap
for prefix in score_prefixes:
    df[f'ratio_{prefix}'] = df[f'{prefix}_a'] / (df[f'{prefix}_b'] + 1e-9)

# Magnitude of difference — does a big gap matter more?
for prefix in score_prefixes:
    df[f'abs_diff_{prefix}'] = np.abs(df[f'diff_{prefix}'])

# Add to feature_cols alongside existing diffs

ratio_cols = [f'ratio_{p}' for p in score_prefixes]
abs_diff_cols = [f'abs_diff_{p}' for p in score_prefixes]
interaction_cols = [f'diff_{p}_x_instr' for p in score_prefixes]

feature_cols = diff_cols + ratio_cols + abs_diff_cols + ['is_instrumental_int'] + interaction_cols
feature_cols = diff_cols + ratio_cols + ['is_instrumental_int'] + interaction_cols
print(feature_cols)
X = df[feature_cols].values

y = (df['user_preference'] == 'A').astype(int).values
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=42, class_weight='balanced'))
])

svm_grid = GridSearchCV(
    svm_pipe,
    param_grid={
        'svc__C': [0.01, 0.1, 1, 10],
        'svc__kernel': ['linear', 'rbf'],
        'svc__gamma': ['scale', 'auto']
    },
    cv=5, scoring='roc_auc', n_jobs=-1
)

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid={
        'n_estimators': [200, 300, 400],
        'max_depth': [3, 4, 5],                    # Stay shallow
        'min_samples_split': [2, 5, 8],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt']
    },
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

from sklearn.model_selection import GridSearchCV

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    param_grid={
        'n_estimators': [100, 300, 500],
        'max_depth': [3, 4, 6],
        'learning_rate': [0.01, 0.05, 0.1],
        'scale_pos_weight': [674/585]
    },
    cv=5, scoring='roc_auc', n_jobs=-1
)


# Voting ensemble of your best models
# from sklearn.ensemble import VotingClassifier

# ensemble = VotingClassifier(
#     estimators=[
#         ('xgb', XGBClassifier(...)),
#         ('rf', RandomForestClassifier(...)),
#         ('lr', Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(...))]))
#     ],
#     voting='soft'  # uses probabilities
# )


# models = {
#     'LogisticRegression': LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced'),
#     'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42, class_weight='balanced'),
#     'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
#                               random_state=42, eval_metric='logloss', verbosity=0, scale_pos_weight=674/585),
#     'AdaBoost': AdaBoostClassifier(n_estimators=300, learning_rate=0.05, random_state=42),
#     'SVM': svm_grid
#     # 'RF grid': rf_grid,
#     # 'xgBoostGrad': xgb_grid
# }

# for name, model in models.items():
#     oof_preds = cross_val_predict(model, X, y, cv=cv)
#     print(f"\n{name}")
#     print(f"  Overall:          {accuracy_score(y, oof_preds):.3f}")
#     print(f"  Instrumental:     {accuracy_score(y[instr_mask], oof_preds[instr_mask]):.3f}")
#     print(f"  Non-instrumental: {accuracy_score(y[~instr_mask], oof_preds[~instr_mask]):.3f}")
#     print(f"  Balanced accuracy: {balanced_accuracy_score(y, oof_preds):.3f}")
#     oof_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
#     cm = confusion_matrix(y, oof_preds)
#     print(f"  Accuracy class B: {cm[0,0] / cm[0,:].sum():.3f}")
#     print(f"  Accuracy class A: {cm[1,1] / cm[1,:].sum():.3f}")

#     print(f"  F1:               {f1_score(y, oof_preds):.3f}")
#     print(f"  AUC:              {roc_auc_score(y, oof_proba):.3f}")
#     print(f"  F1 Instrumental:          {f1_score(y[instr_mask], oof_preds[instr_mask]):.3f}")
#     print(f"  F1 Non-instrumental:      {f1_score(y[~instr_mask], oof_preds[~instr_mask]):.3f}")
#     print(f"  AUC Instrumental:         {roc_auc_score(y[instr_mask], oof_proba[instr_mask]):.3f}")
#     print(f"  AUC Non-instrumental:     {roc_auc_score(y[~instr_mask], oof_proba[~instr_mask]):.3f}")
#     # print(classification_report(y, oof_preds, target_names=['Prefers B', 'Prefers A']))
    
#     fold_accs = []
#     for train_idx, val_idx in cv.split(X, y):
#         model.fit(X[train_idx], y[train_idx])
#         fold_accs.append(accuracy_score(y[val_idx], model.predict(X[val_idx])))
#     print(f"  Per-fold: {np.mean(fold_accs):.3f} ± {np.std(fold_accs):.3f}")

results = []







# ==================== RULE-BASED MODELS ====================

# 1. Likes Rule
rule_preds_likes = (df['pred_likes_a'] < df['pred_likes_b']).astype(int).values
margin_likes = df['pred_likes_b'] - df['pred_likes_a']
rule_proba_likes = 1 / (1 + np.exp(-margin_likes / 10))

results.append({
    'Model': 'Likes (rule)',
    'AUC_overall': round(roc_auc_score(y, rule_proba_likes), 3),
    'AUC_instrumental': round(roc_auc_score(y[instr_mask], rule_proba_likes[instr_mask]), 3),
    'AUC_noninstr': round(roc_auc_score(y[~instr_mask], rule_proba_likes[~instr_mask]), 3),
    'F1_overall': round(f1_score(y, rule_preds_likes), 3),
    'F1_instrumental': round(f1_score(y[instr_mask], rule_preds_likes[instr_mask]), 3),
    'F1_noninstr': round(f1_score(y[~instr_mask], rule_preds_likes[~instr_mask]), 3),
})

# 2. Streams Rule
rule_preds_streams = (df['pred_streams_a'] < df['pred_streams_b']).astype(int).values
margin_streams = df['pred_streams_b'] - df['pred_streams_a']
rule_proba_streams = 1 / (1 + np.exp(-margin_streams / 10))

results.append({
    'Model': 'Streams (rule)',
    'AUC_overall': round(roc_auc_score(y, rule_proba_streams), 3),
    'AUC_instrumental': round(roc_auc_score(y[instr_mask], rule_proba_streams[instr_mask]), 3),
    'AUC_noninstr': round(roc_auc_score(y[~instr_mask], rule_proba_streams[~instr_mask]), 3),
    'F1_overall': round(f1_score(y, rule_preds_streams), 3),
    'F1_instrumental': round(f1_score(y[instr_mask], rule_preds_streams[instr_mask]), 3),
    'F1_noninstr': round(f1_score(y[~instr_mask], rule_preds_streams[~instr_mask]), 3),
})

# 3. Likes + Streams Combined
combined_margin_ls = (margin_likes + margin_streams) / 2
rule_preds_ls = (combined_margin_ls > 0).astype(int).values
rule_proba_ls = 1 / (1 + np.exp(-combined_margin_ls / 10))

results.append({
    'Model': 'Likes + Streams (rule)',
    'AUC_overall': round(roc_auc_score(y, rule_proba_ls), 3),
    'AUC_instrumental': round(roc_auc_score(y[instr_mask], rule_proba_ls[instr_mask]), 3),
    'AUC_noninstr': round(roc_auc_score(y[~instr_mask], rule_proba_ls[~instr_mask]), 3),
    'F1_overall': round(f1_score(y, rule_preds_ls), 3),
    'F1_instrumental': round(f1_score(y[instr_mask], rule_preds_ls[instr_mask]), 3),
    'F1_noninstr': round(f1_score(y[~instr_mask], rule_preds_ls[~instr_mask]), 3),
})

# 4. All Features Rule (Strong Baseline)
score_prefixes = [
    'pred_streams', 'pred_likes', 'pred_coherence', 'pred_musicality',
    'pred_memorability', 'pred_clarity', 'pred_naturalness']

margins = []
for prefix in score_prefixes:
    margin = df[f'{prefix}_b'] - df[f'{prefix}_a']
    margins.append(margin)

all_margin = np.mean(margins, axis=0)                    # shape: (n_samples,)

rule_preds_all = (all_margin > 0).astype(int)            # Removed .values
rule_proba_all = 1 / (1 + np.exp(-all_margin / 10))

results.append({
    'Model': 'All Features (rule)',
    'AUC_overall': round(roc_auc_score(y, rule_proba_all), 3),
    'AUC_instrumental': round(roc_auc_score(y[instr_mask], rule_proba_all[instr_mask]), 3),
    'AUC_noninstr': round(roc_auc_score(y[~instr_mask], rule_proba_all[~instr_mask]), 3),
    'F1_overall': round(f1_score(y, rule_preds_all), 3),
    'F1_instrumental': round(f1_score(y[instr_mask], rule_preds_all[instr_mask]), 3),
    'F1_noninstr': round(f1_score(y[~instr_mask], rule_preds_all[~instr_mask]), 3),
})


# 4. Aesthetics only
score_prefixes = [
    'pred_coherence', 'pred_musicality',
    'pred_memorability', 'pred_clarity', 'pred_naturalness']

margins = []
for prefix in score_prefixes:
    margin = df[f'{prefix}_b'] - df[f'{prefix}_a']
    margins.append(margin)

all_margin = np.mean(margins, axis=0)                    # shape: (n_samples,)

rule_preds_all = (all_margin > 0).astype(int)            # Removed .values
rule_proba_all = 1 / (1 + np.exp(-all_margin / 10))

results.append({
    'Model': 'Aesthetics (rule)',
    'AUC_overall': round(roc_auc_score(y, rule_proba_all), 3),
    'AUC_instrumental': round(roc_auc_score(y[instr_mask], rule_proba_all[instr_mask]), 3),
    'AUC_noninstr': round(roc_auc_score(y[~instr_mask], rule_proba_all[~instr_mask]), 3),
    'F1_overall': round(f1_score(y, rule_preds_all), 3),
    'F1_instrumental': round(f1_score(y[instr_mask], rule_preds_all[instr_mask]), 3),
    'F1_noninstr': round(f1_score(y[~instr_mask], rule_preds_all[~instr_mask]), 3),
})





for name, model in models.items():
    oof_preds = cross_val_predict(model, X, y, cv=cv)
    oof_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]

    results.append({
        'Model': name,
        'AUC_overall':        round(roc_auc_score(y, oof_proba), 3),
        'AUC_instrumental':   round(roc_auc_score(y[instr_mask], oof_proba[instr_mask]), 3),
        'AUC_noninstr':       round(roc_auc_score(y[~instr_mask], oof_proba[~instr_mask]), 3),
        'F1_overall':         round(f1_score(y, oof_preds), 3),
        'F1_instrumental':    round(f1_score(y[instr_mask], oof_preds[instr_mask]), 3),
        'F1_noninstr':        round(f1_score(y[~instr_mask], oof_preds[~instr_mask]), 3),
    })



# rf_grid.fit(X, y)   # ← Fit once on full data

# print("✅ Best params:", rf_grid.best_params_)
# print("✅ Best CV AUC:", round(rf_grid.best_score_, 4))

# if hasattr(rf_grid, 'best_params_'):
#     print(f"\n✅ RandomForest - Best Params:")
#     for param, value in rf_grid.best_params_.items():
#         print(f"   {param}: {value}")
#     print(f"   Best CV AUC: {rf_grid.best_score_:.4f}")


results_df = pd.DataFrame(results).set_index('Model')
print(results_df.to_string())

# Export to Excel
results_df.to_csv('model_results.csv')

# Export to LaTeX
print("\n--- LaTeX ---")
print(results_df.to_latex(float_format='%.3f'))

['diff_pred_streams', 'diff_pred_likes', 'diff_combined_popularity', 'ratio_pred_streams', 'ratio_pred_likes', 'ratio_combined_popularity', 'is_instrumental_int', 'diff_pred_streams_x_instr', 'diff_pred_likes_x_instr', 'diff_combined_popularity_x_instr']
                        AUC_overall  AUC_instrumental  AUC_noninstr  F1_overall  F1_instrumental  F1_noninstr
Model                                                                                                        
Likes (rule)                  0.518             0.500         0.562       0.476            0.454        0.513
Streams (rule)                0.557             0.598         0.482       0.509            0.533        0.470
Likes + Streams (rule)        0.534             0.539         0.535       0.498            0.491        0.509
All Features (rule)           0.535             0.540         0.536       0.499            0.484        0.524
Aesthetics (rule)             0.527             0.531         0.519       0.498      